# Regresión lineal múltiple: precio de viviendas con varias variables

En este notebook vamos a:

1. Comparar la regresión simple con la múltiple y ver por qué **cambian los coeficientes** al añadir variables.
2. Leer el `summary` de statsmodels de un modelo con varias variables.
3. Interpretar cada coeficiente **ceteris paribus**: manteniendo constantes las demás variables.
4. Detectar la **multicolinealidad** con la matriz de correlaciones y el VIF.
5. **Comparar modelos** con $R^2$ ajustado, AIC, BIC y el contraste F de modelos anidados.
6. Revisar los **residuos** del modelo elegido.
7. **Predecir** el precio de una vivienda nueva con su intervalo, y ver cuándo la predicción es una extrapolación.

El modelo de regresión lineal múltiple con $p$ variables explicativas es

$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_p X_p + \varepsilon,$$

donde cada $\beta_j$ es el cambio medio de $Y$ cuando $X_j$ aumenta en una unidad **y las demás variables no cambian**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.anova import anova_lm

## 1. Los datos: quince viviendas

Una inmobiliaria tiene quince viviendas vendidas en la misma ciudad. De cada una conoce:

- `superficie`: metros cuadrados;
- `antiguedad`: años desde su construcción;
- `distancia`: kilómetros al centro;
- `precio`: precio de venta, en miles de euros.

In [ ]:
viviendas = pd.DataFrame(
    {
        "superficie": [75, 90, 100, 85, 95, 70, 110, 80, 88, 92, 78, 105, 82, 96, 89],
        "antiguedad": [19, 17, 22, 17, 8, 5, 22, 12, 15, 19, 5, 21, 11, 15, 12],
        "distancia": [2.0, 3.0, 1.5, 2.5, 5.0, 6.0, 1.0, 4.0, 3.5, 2.2, 5.5, 1.8, 4.5, 3.2, 3.8],
        "precio": [211, 250, 293, 238, 229, 157, 320, 197, 234, 247, 176, 295, 194, 251, 223],
    }
)

viviendas.describe().round(2)

Dibujamos el precio frente a cada variable por separado.

In [ ]:
etiquetas = {
    "superficie": "Superficie (m²)",
    "antiguedad": "Antigüedad (años)",
    "distancia": "Distancia al centro (km)",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (var, etiqueta) in zip(axes, etiquetas.items()):
    ax.scatter(viviendas[var], viviendas["precio"])
    ax.set_xlabel(etiqueta)
axes[0].set_ylabel("Precio (miles €)")
plt.show()

Las tres variables están relacionadas con el precio: las viviendas más grandes y las más cercanas al centro son más caras, como cabía esperar. Lo que sorprende es el gráfico del centro: **las viviendas más antiguas también parecen más caras**.

## 2. Regresión simple frente a regresión múltiple

### Una variable cada vez

Ajustamos tres regresiones simples, una por variable.

In [ ]:
filas = []
for var in ["superficie", "antiguedad", "distancia"]:
    m = smf.ols(f"precio ~ {var}", data=viviendas).fit()
    filas.append(
        {
            "variable": var,
            "pendiente": m.params[var],
            "p-valor": m.pvalues[var],
            "R²": m.rsquared,
        }
    )
pd.DataFrame(filas).round({"pendiente": 3, "R²": 4})

Según la regresión simple, cada año más de antigüedad se asocia con unos **6 750 € más** de precio. ¿De verdad una vivienda vale más por ser más vieja? Antes de creerlo, miramos cómo se relacionan las variables explicativas entre sí.

In [ ]:
plt.scatter(viviendas["distancia"], viviendas["antiguedad"])
plt.xlabel("Distancia al centro (km)")
plt.ylabel("Antigüedad (años)")
plt.title("Las viviendas antiguas están en el centro")
plt.show()

print(f"Correlación entre antigüedad y distancia: {viviendas['antiguedad'].corr(viviendas['distancia']):.3f}")

En esta ciudad, las viviendas antiguas están en el centro y las nuevas, en las afueras: la correlación es $-0.99$. La regresión simple del precio sobre la antigüedad no puede separar las dos cosas, así que atribuye a la antigüedad lo que en realidad se debe, al menos en parte, a la **ubicación**. La antigüedad hace de "representante" de la distancia.

Es el problema de la **variable omitida**: si una variable que influye en $Y$ (la distancia) está correlacionada con la que sí está en el modelo (la antigüedad), el coeficiente de esta recoge también el efecto de la que falta.

### El modelo con las tres variables

En la fórmula, las variables se suman con `+`.

In [ ]:
modelo = smf.ols("precio ~ superficie + antiguedad + distancia", data=viviendas).fit()
print(modelo.summary())

Comparamos la pendiente de cada variable en la regresión simple y en la múltiple:

In [ ]:
comparacion = pd.DataFrame(
    {
        "regresión simple": [
            smf.ols(f"precio ~ {v}", data=viviendas).fit().params[v]
            for v in ["superficie", "antiguedad", "distancia"]
        ],
        "regresión múltiple": modelo.params[["superficie", "antiguedad", "distancia"]],
    }
)
comparacion.round(3)

- **Superficie:** baja de 3.80 a 2.67 mil € por m². En la regresión simple, la superficie recogía también parte del efecto de la ubicación, porque en estos datos las viviendas grandes tienden a estar más cerca del centro (correlación entre superficie y distancia de $-0.66$). Con la distancia en el modelo, la superficie ya no se lleva ese efecto.
- **Antigüedad:** baja de 6.75 a 2.02, y deja de ser significativa.
- **Distancia:** pasa de $-25.7$ a $-5.0$, y tampoco es significativa.

Que un coeficiente cambie al añadir variables no es un error: los dos responden a **preguntas distintas**. La regresión simple responde a "¿cuánto más caras son, en promedio, las viviendas de 1 m² más?". La múltiple responde a "¿cuánto más cara es una vivienda con 1 m² más, **con la misma antigüedad y la misma distancia**?".

## 3. Cómo leer el `summary` de un modelo múltiple

El `summary` tiene tres bloques.

**Bloque superior: el modelo en conjunto**

- `R-squared`: proporción de la variabilidad del precio que explica el modelo (0.987). Con más variables nunca baja, aunque la variable añadida no sirva para nada.
- `Adj. R-squared`: el $R^2$ ajustado, que penaliza el número de variables. Es el que se usa para comparar modelos con distinto número de variables.
- `F-statistic` y `Prob (F-statistic)`: el **contraste F global**, $H_0: \beta_1 = \beta_2 = \beta_3 = 0$ (ninguna variable sirve). Aquí $F = 270$ con un p-valor de $1.4 \times 10^{-10}$: el modelo, en conjunto, explica el precio.
- `AIC` y `BIC`: criterios de información para comparar modelos (sección 6). Menor es mejor.
- `Df Residuals`: $n - p - 1 = 15 - 3 - 1 = 11$ grados de libertad.

**Bloque central: un contraste t por coeficiente**

Cada fila contrasta $H_0: \beta_j = 0$ **con las demás variables en el modelo**. La superficie es claramente significativa ($t = 14.2$). La antigüedad ($p = 0.31$) y la distancia ($p = 0.50$) no lo son, y sus intervalos de confianza incluyen el 0 y valores muy distintos entre sí.

**Bloque inferior: los residuos y un aviso**

`Omnibus` y `Jarque-Bera` contrastan la normalidad de los residuos (con 15 datos dicen poco). `Cond. No.` es el número de condición: si es grande, statsmodels añade la nota *"This might indicate that there are strong multicollinearity"*.

### Una contradicción aparente

El contraste F dice que el modelo explica muy bien el precio ($R^2 = 0.987$), pero dos de las tres variables no son significativas por separado. ¿Cómo puede ser? Porque la antigüedad y la distancia llevan casi la misma información: juntas explican mucho, pero el modelo no sabe a cuál de las dos atribuirlo. Es el síntoma típico de la **multicolinealidad**, que medimos en la sección 5.

## 4. Interpretar los coeficientes: ceteris paribus

*Ceteris paribus* significa "lo demás constante". Cada coeficiente de la regresión múltiple es el cambio medio del precio cuando su variable aumenta en una unidad **y las otras se quedan igual**.

Lo comprobamos con dos viviendas que solo se diferencian en 10 m²:

In [ ]:
dos_viviendas = pd.DataFrame(
    {"superficie": [85, 95], "antiguedad": [15, 15], "distancia": [3.0, 3.0]}
)
pred = modelo.predict(dos_viviendas)

print(dos_viviendas.assign(precio_predicho=pred.round(2)))
print(f"\nDiferencia: {pred[1] - pred[0]:.2f} mil €  =  10 × {modelo.params['superficie']:.3f}")

La diferencia de precio predicha es exactamente $10 \times \hat{\beta}_{\text{superficie}}$, sea cual sea la antigüedad y la distancia que fijemos.

Las interpretaciones, en las unidades del problema:

- **Superficie:** con la misma antigüedad y la misma distancia al centro, cada m² más se asocia, en promedio, con **2 670 € más** de precio (intervalo al 95 %: de 2 260 a 3 080 €).
- **Antigüedad:** con la misma superficie y distancia, cada año más se asocia con 2 020 € más, pero el intervalo va de $-2\,160$ a $+6\,200$ €: los datos **no permiten saber** si la antigüedad, a igual ubicación, sube o baja el precio.
- **Distancia:** con la misma superficie y antigüedad, cada km más lejos del centro se asocia con 5 010 € menos, con un intervalo de $-20\,970$ a $+10\,940$ €: tampoco se puede concluir nada.
- **Intercepto:** sería el precio de una vivienda de 0 m², 0 años y a 0 km del centro. No tiene interpretación práctica: está muy lejos de los datos.

¿Por qué no se puede concluir nada de la antigüedad ni de la distancia? Para separar sus efectos haría falta ver viviendas antiguas lejos del centro, o nuevas en el centro, y en estos datos casi no las hay. "Mantener la distancia constante y cambiar la antigüedad" es una comparación que los datos no contienen.

## 5. Multicolinealidad: matriz de correlaciones y VIF

### Matriz de correlaciones

Es lo primero que hay que mirar antes de ajustar un modelo múltiple.

In [ ]:
correlaciones = viviendas.corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(correlaciones, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(correlaciones)), correlaciones.columns, rotation=45)
ax.set_yticks(range(len(correlaciones)), correlaciones.columns)
for i in range(len(correlaciones)):
    for j in range(len(correlaciones)):
        ax.text(j, i, f"{correlaciones.iloc[i, j]:.2f}", ha="center", va="center")
fig.colorbar(im)
ax.set_title("Matriz de correlaciones")
plt.show()

La correlación de $-0.99$ entre antigüedad y distancia es la que causa el problema. La superficie también está correlacionada con las otras dos (unos $\pm 0.66$), pero mucho menos.

### El factor de inflación de la varianza (VIF)

La matriz de correlaciones mira las variables de dos en dos. El **VIF** de una variable $X_j$ mide cuánto la explican **todas las demás juntas**:

$$\text{VIF}_j = \frac{1}{1 - R_j^2},$$

donde $R_j^2$ es el $R^2$ de la regresión de $X_j$ sobre las demás variables explicativas. Si $R_j^2 = 0$, el VIF vale 1 (no hay multicolinealidad); cuanto más se acerca $R_j^2$ a 1, más crece el VIF. Su raíz cuadrada dice cuántas veces es mayor el error estándar de $\hat{\beta}_j$ de lo que sería sin multicolinealidad. Como orientación: un VIF por encima de 5 merece atención y por encima de 10 es preocupante.

In [ ]:
X = sm.add_constant(viviendas[["superficie", "antiguedad", "distancia"]])
vif = pd.Series(
    [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])],
    index=X.columns[1:],
    name="VIF",
)
vif.round(2)

La constante se añade a `X` para calcular bien los $R_j^2$, pero su VIF no se interpreta (por eso el bucle empieza en 1).

Comprobamos el VIF de la antigüedad a mano:

In [ ]:
r2_antiguedad = smf.ols("antiguedad ~ superficie + distancia", data=viviendas).fit().rsquared
print(f"R² de antigüedad sobre las demás: {r2_antiguedad:.4f}")
print(f"VIF = 1 / (1 - {r2_antiguedad:.4f}) = {1 / (1 - r2_antiguedad):.2f}")
print(f"Raíz del VIF: {np.sqrt(1 / (1 - r2_antiguedad)):.1f}")

La superficie y la distancia explican el 98 % de la variación de la antigüedad. El VIF de 48 multiplica el error estándar de su coeficiente por casi 7: por eso sus intervalos son tan anchos.

La multicolinealidad **no estropea las predicciones** del modelo dentro del rango de los datos (el $R^2$ sigue siendo 0.987), pero hace que los coeficientes de las variables implicadas sean imprecisos e inestables: con otra muestra podrían cambiar mucho, incluso de signo. Si el objetivo es interpretar los coeficientes, conviene quedarse con una de las dos variables. Las secciones siguientes ayudan a decidir cuál.

## 6. Comparar modelos

Comparamos tres modelos **anidados** (cada uno contiene al anterior):

1. `precio ~ superficie`
2. `precio ~ superficie + distancia`
3. `precio ~ superficie + antiguedad + distancia`

Añadimos un cuarto, `precio ~ superficie + antiguedad`, que cambia la distancia por la antigüedad.

In [ ]:
formulas = {
    "1. superficie": "precio ~ superficie",
    "2. superficie + distancia": "precio ~ superficie + distancia",
    "3. las tres": "precio ~ superficie + antiguedad + distancia",
    "4. superficie + antigüedad": "precio ~ superficie + antiguedad",
}
modelos = {nombre: smf.ols(f, data=viviendas).fit() for nombre, f in formulas.items()}

tabla = pd.DataFrame(
    {
        nombre: {
            "variables": int(m.df_model),
            "R²": m.rsquared,
            "R² ajustado": m.rsquared_adj,
            "AIC": m.aic,
            "BIC": m.bic,
            "error típico s": np.sqrt(m.scale),
        }
        for nombre, m in modelos.items()
    }
).T
tabla.round(4)

Cómo se lee cada columna:

- **$R^2$** sube siempre al añadir variables (de 0.9852 a 0.9866 al pasar del modelo 2 al 3), así que no sirve para comparar modelos con distinto número de variables.
- **$R^2$ ajustado** penaliza cada variable añadida. Pasar de la superficie sola a superficie y distancia lo sube de 0.877 a 0.983. Añadir después la antigüedad apenas lo mueve (0.9828 frente a 0.9830): la mejora es mínima.
- **AIC y BIC** también premian el ajuste y castigan las variables (el BIC castiga más). Los dos son menores, es decir, mejores, en el modelo 2 que en el 3.
- El **error típico** $s$ es el tamaño típico de los residuos, en miles de €: baja de 15.8 a 5.9 al añadir la distancia y no mejora al añadir la antigüedad.

### Contraste F de modelos anidados

¿La mejora al añadir variables es mayor de lo que se esperaría por azar? El contraste F compara dos modelos anidados: $H_0$ dice que los coeficientes de las variables añadidas son 0. `anova_lm` lo calcula para una cadena de modelos, cada uno frente al anterior.

In [ ]:
anova_lm(modelos["1. superficie"], modelos["2. superficie + distancia"], modelos["3. las tres"]).round(4)

Cada fila compara el modelo con el de la fila anterior. `ssr` es la suma de los residuos al cuadrado y `ss_diff`, cuánto la reduce el modelo más grande.

- **Fila 1 (añadir la distancia a la superficie):** la suma de residuos al cuadrado baja de 3 238 a 417, $F = 82$ y $p \approx 10^{-6}$. La distancia aporta mucho.
- **Fila 2 (añadir la antigüedad a superficie y distancia):** baja de 417 a 378, $F = 1.13$ y $p = 0.31$. No hay evidencia de que la antigüedad aporte algo cuando ya se conocen la superficie y la distancia.

### ¿Distancia o antigüedad?

El modelo 4 (superficie y antigüedad) sale incluso algo mejor en los números: $R^2$ ajustado de 0.9837 frente a 0.9828, AIC de 97.6 frente a 98.5 y BIC de 99.7 frente a 100.6. Pero las diferencias de AIC y BIC son menores que 1, y eso es un **empate**: con quince viviendas, una diferencia así puede deberse al azar. Los dos modelos se ajustan casi igual porque en estos datos la antigüedad y la distancia llevan casi la misma información (correlación de $-0.99$). Los números no deciden entre los dos; decide el conocimiento del problema. La ubicación es una razón conocida y directa del precio de una vivienda; que la antigüedad suba el precio no lo es. En el modelo 4, el coeficiente positivo de la antigüedad se leería como "cada año más, 3 300 € más", una conclusión engañosa: la antigüedad solo está haciendo de representante de la ubicación.

**Elegimos el modelo 2: `precio ~ superficie + distancia`.**

In [ ]:
elegido = modelos["2. superficie + distancia"]
print(elegido.summary().tables[1])

Con la misma distancia al centro, cada m² más se asocia con unos **2 670 € más** de precio. Con la misma superficie, cada km más lejos del centro se asocia con unos **12 600 € menos** (intervalo: de 9 500 a 15 600 € menos). Ahora los dos intervalos son estrechos: sin la antigüedad, ya no hay multicolinealidad.

In [ ]:
X2 = sm.add_constant(viviendas[["superficie", "distancia"]])
pd.Series(
    [variance_inflation_factor(X2.values, i) for i in range(1, X2.shape[1])],
    index=X2.columns[1:],
    name="VIF",
).round(2)

## 7. Diagnóstico de los residuos

Las conclusiones del modelo (p-valores, intervalos) se apoyan en que los errores tienen media cero, varianza constante, son independientes y siguen aproximadamente una normal. Lo comprobamos con los residuos del modelo elegido:

- **Residuos frente a valores ajustados:** no debería haber curva (falta de linealidad) ni forma de embudo (varianza no constante).
- **Residuos frente a cada variable explicativa:** una curva indicaría que esa variable no entra de forma lineal.
- **Gráfico Q-Q:** los puntos deberían seguir la diagonal si los residuos son aproximadamente normales.

In [ ]:
residuos = elegido.resid
ajustados = elegido.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].scatter(ajustados, residuos)
axes[0, 0].axhline(0, color="red", ls="--")
axes[0, 0].set_xlabel("Precio ajustado (miles €)")
axes[0, 0].set_ylabel("Residuo")
axes[0, 0].set_title("Residuos frente a valores ajustados")

for ax, var in zip([axes[0, 1], axes[1, 0]], ["superficie", "distancia"]):
    ax.scatter(viviendas[var], residuos)
    ax.axhline(0, color="red", ls="--")
    ax.set_xlabel(etiquetas[var])
    ax.set_ylabel("Residuo")
    ax.set_title(f"Residuos frente a {var}")

stats.probplot(residuos, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("Gráfico Q-Q de los residuos")

plt.tight_layout()
plt.show()

Ningún gráfico muestra curvas ni embudos, y el Q-Q sigue la diagonal. Con quince viviendas solo se pueden detectar problemas graves, y no los hay.

También conviene mirar si alguna vivienda tiene demasiado peso en el ajuste. La **distancia de Cook** mide cuánto cambiarían las predicciones si se quitara esa observación; como orientación, valores por encima de 1 (o muy por encima del resto) merecen una revisión.

In [ ]:
influencia = elegido.get_influence()
diagnostico = viviendas.assign(
    residuo=residuos.round(2),
    residuo_estudentizado=influencia.resid_studentized_external.round(2),
    cook=influencia.cooks_distance[0].round(3),
)
diagnostico.sort_values("cook", ascending=False).head(5)

La vivienda más influyente es la de 70 m² a 6 km del centro (la más pequeña y la más lejana), con una distancia de Cook de 0.39, lejos de 1. Ningún residuo estudentizado pasa de 2 en valor absoluto. No hay observaciones problemáticas.

Por último, dos contrastes formales: Shapiro-Wilk para la normalidad y Breusch-Pagan para la varianza constante. En los dos, $H_0$ es que el supuesto se cumple.

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

shapiro = stats.shapiro(residuos)
bp_estadistico, bp_p, _, _ = het_breuschpagan(residuos, elegido.model.exog)

print(f"Shapiro-Wilk:   p-valor = {shapiro.pvalue:.3f}")
print(f"Breusch-Pagan:  p-valor = {bp_p:.3f}")

Los dos p-valores son altos: no hay evidencia contra la normalidad ni contra la varianza constante. Con muestras tan pequeñas, estos contrastes tienen poca potencia, así que los gráficos pesan tanto como ellos.

## 8. Predicción para una vivienda nueva

La inmobiliaria quiere tasar una vivienda de **85 m², 7 años y a 2.5 km del centro**. `get_prediction` da dos intervalos:

- `mean_ci`: el intervalo de confianza del precio **medio** de todas las viviendas con esas características;
- `obs_ci`: el intervalo de predicción del precio de **esta** vivienda, más ancho porque añade la variación de una vivienda concreta alrededor de la media.

In [ ]:
nueva = pd.DataFrame({"superficie": [85], "antiguedad": [7], "distancia": [2.5]})

prediccion = elegido.get_prediction(nueva).summary_frame(alpha=0.05)
prediccion.round(2)

Con el modelo elegido, el precio predicho es de unos **233 700 €**, con un intervalo de predicción al 95 % de **219 900 a 247 500 €**. El modelo elegido no usa la antigüedad, así que la columna `antiguedad` de `nueva` se ignora.

### Cuidado: extrapolación escondida

Cada valor de la vivienda nueva está dentro del rango de los datos: 85 m² (de 70 a 110), 7 años (de 5 a 22) y 2.5 km (de 1 a 6). Pero la **combinación** no lo está: es una vivienda **nueva y céntrica**, y en los datos todas las viviendas céntricas son antiguas: las vendidas a entre 2 y 3.2 km del centro tienen de 15 a 19 años.

In [ ]:
plt.scatter(viviendas["distancia"], viviendas["antiguedad"], label="viviendas vendidas")
plt.scatter(nueva["distancia"], nueva["antiguedad"], color="C3", marker="*", s=250, label="vivienda nueva")
plt.xlabel("Distancia al centro (km)")
plt.ylabel("Antigüedad (años)")
plt.legend()
plt.show()

¿Qué cambia? El modelo elegido supone que, a igual superficie y distancia, la antigüedad no afecta al precio. Con estos datos no lo podemos comprobar para una vivienda nueva en el centro, porque no hay ninguna. El modelo con las tres variables lo hace visible: su predicción es muy distinta y su intervalo, mucho más ancho.

In [ ]:
pd.concat(
    {
        "superficie + distancia": elegido.get_prediction(nueva).summary_frame(),
        "las tres variables": modelos["3. las tres"].get_prediction(nueva).summary_frame(),
    }
).round(2)

Con las tres variables, el precio predicho baja a unos 212 000 € y el intervalo de predicción va de 165 500 a 258 800 €: más de tres veces más ancho. El error estándar de la media predicha (`mean_se`) pasa de 2.3 a 20.4 mil €. El modelo está avisando de que esa combinación de antigüedad y distancia queda lejos de las viviendas con las que se ha ajustado.

Con una multicolinealidad tan fuerte entre antigüedad y distancia, esta predicción **no es fiable**, la dé un modelo o el otro.

**Qué decirle a la inmobiliaria:** el modelo da una referencia de unos 234 000 €, pero la vivienda es de un tipo (obra nueva en el centro) que no aparece en los datos. Si la obra nueva tiene un precio distinto a igual ubicación, el modelo no lo puede saber. Para tasarla con confianza harían falta ventas de viviendas nuevas céntricas.

### La extrapolación evidente

Una vivienda de 150 m² queda fuera del rango de superficies observado (de 70 a 110 m²). El modelo da un número, pero nada en los datos garantiza que la relación siga siendo lineal tan lejos.

In [ ]:
grande = pd.DataFrame({"superficie": [150], "distancia": [2.5]})
elegido.get_prediction(grande).summary_frame().round(2)

Los intervalos se ensanchan, pero solo reflejan la incertidumbre **si el modelo lineal sigue valiendo** a 150 m². No protegen contra que la relación cambie fuera de los datos, por ejemplo si el precio por m² baja en las viviendas muy grandes.

Para detectar la extrapolación hay que comparar la vivienda nueva con los datos: cada variable con su rango y, cuando hay varias, **la combinación** con las que se han observado (con un gráfico de dispersión de las variables explicativas, como el de arriba).

## 9. Resumen

- Un coeficiente de regresión múltiple es el efecto de su variable **con las demás constantes**. Por eso puede ser muy distinto del de la regresión simple: responden a preguntas distintas.
- Una variable omitida que está correlacionada con una incluida contamina su coeficiente: la antigüedad parecía subir el precio porque las viviendas antiguas son céntricas.
- El contraste F global y los contrastes t pueden discrepar: si el modelo es significativo pero sus variables no lo son por separado, sospecha de multicolinealidad y mira la matriz de correlaciones y el VIF.
- Para comparar modelos con distinto número de variables, usa $R^2$ ajustado, AIC y BIC, y el contraste F de modelos anidados; cuando los números empatan, decide el conocimiento del problema.
- Antes de predecir, comprueba que la combinación de valores se parece a los datos: cada variable dentro de su rango no basta.

## 10. Para pensar

1. Ajusta `precio ~ antiguedad + distancia`. ¿Qué signo tiene la antigüedad? ¿Y su VIF?
2. En la sección 4, cambia la antigüedad y la distancia de las dos viviendas (por ejemplo, 5 años y 1 km). ¿Cambia la diferencia de precio predicha? ¿Por qué?
3. Añade a los datos una vivienda nueva y céntrica: 85 m², 6 años, 2 km y 255 mil €. Ajusta de nuevo el modelo con las tres variables. ¿Qué les pasa al VIF de la antigüedad y al error estándar de su coeficiente?
4. ¿Qué modelo elegirías si el único objetivo fuera predecir el precio de viviendas parecidas a las de los datos? ¿Cambia tu respuesta si el objetivo es explicar a un cliente qué hace subir el precio?